# Proyecto de Metodos Numericos: Modelado de Exito de Videojuegos (Steam)

Institucion: Escuela Politecnica Nacional (EPN)  
Materia: Metodos Numericos  
Semestre: 2026-A  
Estudiantes:  
* Ricardo Cisneros  
* Hugo Coyago  
* Jair Paez  

---  

# Parte 1: Creacion, Unificacion y Estructuracion del Dataset

Para garantizar la replicabilidad absoluta de este proyecto cientifico, la primera etapa consiste en reconstruir el dataset unificado completo directamente desde los datos crudos extraidos de la API de Steam y SteamSpy (almacenados en la carpeta 01_Datos/Datos crudos/steam-insights-main).

### Fuentes de Informacion Utilizadas
1. games.csv: Metadatos de la tienda oficial de Steam (nombre, fecha de lanzamiento, tipo de producto, etc.).
2. reviews.csv: Estadisticas de valoraciones y resenas de usuarios publicas.
3. steamspy_insights.csv: Datos de consumo, precios en centavos y rangos de compradores estimativos.

### Formulas Matematicas de Estimacion Continua
Para transformar las categorias cualitativas o discretas en variables cuantitativas continuas, implementamos dos ecuaciones clave:

1. Puntaje de Aprobacion Continuo :
   $$A_R = \frac{\text{Resenas Positivas}}{\text{Resenas Positivas} + \text{Resenas Negativas}}$$
   Genera una escala de [0.0, 1.0] que representa el porcentaje real de satisfaccion del jugador.

2. Ventas Estimadas Continuas :
   Dado que SteamSpy reporta las ventas en rangos categoricos [L, U] (Limite Inferior y Superior), calculamos su punto medio aritmetico:
   $$E_O = \frac{L + U}{2}$$
   Ejemplo: Para un rango de 10M a 20M de copias, calculamos un punto medio continuo de 15M.

In [ ]:
import os
import csv

directorio_base = ".."
juegos_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "games", "games.csv")
resenas_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "reviews", "reviews.csv")
steamspy_csv = os.path.join(directorio_base, "01_Datos", "Datos crudos", "steam-insights-main", "steamspy_insights", "steamspy_insights.csv")
salida_datos_unificados_crudos = os.path.join(directorio_base, "01_Datos", "steam_raw_unified_all_columns.csv")

def analizar_rango_duenos(cadena_duenos):
    if not cadena_duenos or '..' not in cadena_duenos:
        return ""
    partes = cadena_duenos.split('..')
    if len(partes) == 2:
        try:
            limite_inferior = int(partes[0].replace(',', '').strip())
            limite_superior = int(partes[1].replace(',', '').strip())
            return (limite_inferior + limite_superior) // 2
        except ValueError:
            return ""
    return ""

datos_juegos = {}
with open(juegos_csv, mode='r', encoding='utf-8') as archivo_juegos:
    lector = csv.DictReader(archivo_juegos)
    for fila in lector:
        id_aplicacion = fila.get("app_id")
        if not id_aplicacion:
            continue
        datos_juegos[id_aplicacion] = {
            "name": fila.get("name", ""),
            "release_date": fila.get("release_date", ""),
            "is_free": fila.get("is_free", ""),
            "price_overview": fila.get("price_overview", ""),
            "games_languages": fila.get("languages", ""),
            "type": fila.get("type", "")
        }

datos_resenas = {}
with open(resenas_csv, mode='r', encoding='utf-8') as archivo_resenas:
    lector = csv.DictReader(archivo_resenas)
    for fila in lector:
        id_aplicacion = fila.get("app_id")
        if not id_aplicacion:
            continue
        datos_resenas[id_aplicacion] = {
            "review_score": fila.get("review_score", ""),
            "review_score_description": fila.get("review_score_description", ""),
            "reviews_positive": fila.get("positive", ""),
            "reviews_negative": fila.get("negative", ""),
            "reviews_total": fila.get("total", ""),
            "metacritic_score": fila.get("metacritic_score", ""),
            "reviews_text": fila.get("reviews", ""),
            "recommendations": fila.get("recommendations", ""),
            "steamspy_user_score": fila.get("steamspy_user_score", ""),
            "steamspy_score_rank": fila.get("steamspy_score_rank", ""),
            "reviews_steamspy_positive": fila.get("steamspy_positive", ""),
            "reviews_steamspy_negative": fila.get("steamspy_negative", "")
        }

datos_steamspy = {}
todos_los_ids_aplicacion = set(datos_juegos.keys()) | set(datos_resenas.keys())
with open(steamspy_csv, mode='r', encoding='utf-8') as archivo_steamspy:
    lector = csv.DictReader(archivo_steamspy)
    for fila in lector:
        id_aplicacion = fila.get("app_id")
        if not id_aplicacion:
            continue
        datos_steamspy[id_aplicacion] = fila
        todos_los_ids_aplicacion.add(id_aplicacion)

todos_los_datos_fusionados = []
for id_aplicacion in sorted(list(todos_los_ids_aplicacion), key=lambda x: int(x) if x.isdigit() else 9999999):
    juego = datos_juegos.get(id_aplicacion, {
        "name": "", "release_date": "", "is_free": "", "price_overview": "", "games_languages": "", "type": ""
    })
    resena = datos_resenas.get(id_aplicacion, {
        "review_score": "", "review_score_description": "", "reviews_positive": "", "reviews_negative": "",
        "reviews_total": "", "metacritic_score": "", "reviews_text": "", "recommendations": "",
        "steamspy_user_score": "", "steamspy_score_rank": "", "reviews_steamspy_positive": "", "reviews_steamspy_negative": ""
    })
    spy = datos_steamspy.get(id_aplicacion, {
        "developer": "", "publisher": "", "owners_range": "", "concurrent_users_yesterday": "",
        "playtime_average_forever": "", "playtime_average_2weeks": "", "playtime_median_forever": "",
        "playtime_median_2weeks": "", "price": "", "initial_price": "", "discount": "", "languages": "", "genres": ""
    })

    precio_crudo = spy.get("price", "")
    precio_dolares = ""
    if precio_crudo and precio_crudo != "\\N":
        try:
            precio_dolares = round(int(precio_crudo) / 100.0, 2)
        except ValueError:
            pass

    positivas_crudas = resena.get("reviews_positive", "")
    negativas_crudas = resena.get("reviews_negative", "")
    proporcion_aprobacion = ""
    if positivas_crudas and negativas_crudas:
        try:
            positivas = int(positivas_crudas)
            negativas = int(negativas_crudas)
            if positivas + negativas > 0:
                proporcion_aprobacion = round(positivas / (positivas + negativas), 4)
        except ValueError:
            pass

    duenos_crudo = spy.get("owners_range", "")
    duenos_estimados = analizar_rango_duenos(duenos_crudo)
    
    fila_fusionada = {
        "app_id": id_aplicacion,
        "name": juego["name"],
        "release_date": juego["release_date"],
        "is_free": juego["is_free"],
        "price_overview": juego["price_overview"],
        "games_languages": juego["games_languages"],
        "type": juego["type"],
        "developer": spy.get("developer", ""),
        "publisher": spy.get("publisher", ""),
        "owners_range": spy.get("owners_range", ""),
        "concurrent_users_yesterday": spy.get("concurrent_users_yesterday", ""),
        "playtime_average_forever": spy.get("playtime_average_forever", ""),
        "playtime_average_2weeks": spy.get("playtime_average_2weeks", ""),
        "playtime_median_forever": spy.get("playtime_median_forever", ""),
        "playtime_median_2weeks": spy.get("playtime_median_2weeks", ""),
        "steamspy_price": spy.get("price", ""),
        "steamspy_initial_price": spy.get("initial_price", ""),
        "steamspy_discount": spy.get("discount", ""),
        "steamspy_languages": spy.get("languages", ""),
        "genres": spy.get("genres", ""),
        "review_score": resena["review_score"],
        "review_score_description": resena["review_score_description"],
        "reviews_positive": resena["reviews_positive"],
        "reviews_negative": resena["reviews_negative"],
        "reviews_total": resena["reviews_total"],
        "metacritic_score": resena["metacritic_score"],
        "reviews_text": resena["reviews_text"],
        "recommendations": resena["recommendations"],
        "steamspy_user_score": resena["steamspy_user_score"],
        "steamspy_score_rank": resena["steamspy_score_rank"],
        "reviews_steamspy_positive": resena["reviews_steamspy_positive"],
        "reviews_steamspy_negative": resena["reviews_steamspy_negative"],
        "price_usd": precio_dolares,
        "approval_ratio": proporcion_aprobacion,
        "estimated_owners": duenos_estimados
    }
    todos_los_datos_fusionados.append(fila_fusionada)

encabezados = [
    "app_id", "name", "release_date", "is_free", "price_overview", "games_languages", "type",
    "developer", "publisher", "owners_range", "concurrent_users_yesterday",
    "playtime_average_forever", "playtime_average_2weeks", "playtime_median_forever", "playtime_median_2weeks",
    "steamspy_price", "steamspy_initial_price", "steamspy_discount", "steamspy_languages", "genres",
    "review_score", "review_score_description", "reviews_positive", "reviews_negative", "reviews_total",
    "metacritic_score", "reviews_text", "recommendations", "steamspy_user_score", "steamspy_score_rank",
    "reviews_steamspy_positive", "reviews_steamspy_negative",
    "price_usd", "approval_ratio", "estimated_owners"
]

with open(salida_datos_unificados_crudos, mode='w', newline='', encoding='utf-8') as archivo_salida:
    escritor = csv.DictWriter(archivo_salida, fieldnames=encabezados)
    escritor.writeheader()
    escritor.writerows(todos_los_datos_fusionados)

print("Consolidacion completada con exito")

---  
## Parte 2: Limpieza de Datos y Ajuste de Curvas